In [127]:
import torch, random

In [128]:
def syn_data(w, b, num):
    X = torch.normal(0, 1, (num, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1, 1)

In [129]:
true_w, true_b = torch.tensor([2, -3.4]), 4.2
features, labels = syn_data(true_w, true_b, 10000)

In [130]:
features[1], labels[1], features.shape, labels.shape

(tensor([-1.1377,  0.0556]),
 tensor([1.7195]),
 torch.Size([10000, 2]),
 torch.Size([10000, 1]))

In [131]:
def data_iter(batch, features, labels):
    num = len(features)
    indices = list(range(num))
    random.shuffle(indices)
    for i in range(0, num, batch):
        batch_indices = indices[i: min(i + batch, num)]
        yield features[batch_indices], labels[batch_indices]

In [132]:
w, b = torch.normal(0, 0.01, size=(2,1), requires_grad=True), torch.zeros(1, requires_grad=True)

In [133]:
def linreg(X, w, b):
    return torch.mm(X, w) + b

In [134]:
def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape))** 2 / 2

In [135]:
def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()
        

In [136]:
lr = 0.03
eps = 10
batch_size = 10

for ep in range(eps):
    for X, y in data_iter(batch_size, features, labels):
        l = squared_loss(linreg(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = squared_loss(linreg(features, w, b), labels)
        print(f'ep {ep + 1}, loss {float(train_l.mean()):.6f}')

ep 1, loss 0.000050
ep 2, loss 0.000050
ep 3, loss 0.000050
ep 4, loss 0.000050
ep 5, loss 0.000050
ep 6, loss 0.000050
ep 7, loss 0.000050
ep 8, loss 0.000050
ep 9, loss 0.000050
ep 10, loss 0.000050


In [137]:
true_w - w.reshape(true_w.shape)

tensor([-0.0004,  0.0007], grad_fn=<SubBackward0>)

In [138]:
true_b - b

tensor([-0.0006], grad_fn=<RsubBackward1>)

In [139]:
import torch
from torch.utils import data
from torch import nn

# 1. 使用 DataLoader 自动加载和打包数据
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# 2. 使用 nn.Sequential 和 nn.Linear 定义网络结构
# nn.Linear(输入特征数, 输出特征数)
net = nn.Sequential(nn.Linear(2, 1))

# 初始化模型参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

# 3. 定义损失函数与优化器
loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 4. 训练循环
epochs = 3
for epoch in range(epochs):
    for X, y in data_iter:
        l = loss(net(X), y)     # 1. 前向传播
        trainer.zero_grad()      # 2. 清空梯度
        l.backward()             # 3. 反向传播
        trainer.step()           # 4. 更新参数
        
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l.item():.6f}')


epoch 1, loss 0.000100
epoch 2, loss 0.000101
epoch 3, loss 0.000100


In [140]:
import torch

# 1. 设定真实的权重 vector 和偏置 scalar
true_w = torch.tensor([-1.5, -2.0], dtype=torch.float64) # 形状: (2,)
true_b = 20.0

# 2. 随机生成 1000 辆车的数据矩阵 X (1000 行, 2 列)
# 第 0 列是里程(0~10万公里)，第 1 列是车龄(0~10年)
X = torch.rand(1000, 2, dtype=torch.float64) * 10

# 3. 计算真实的房价 y (矢量化矩阵乘法)
# X (1000, 2) @ true_w (2,) -> (1000,) 的向量，再加上标量 true_b
# 另外加入一点随机噪声 (高斯分布)
y = X @ true_w + true_b + torch.randn(1000, dtype=torch.float64) * 0.1

print("前 3 辆车的特征 X:\n", X[:3])
print("前 3 辆车的价格 y:\n", y[:3])


前 3 辆车的特征 X:
 tensor([[7.9161, 4.6568],
        [0.2338, 3.3453],
        [5.6575, 2.2366]], dtype=torch.float64)
前 3 辆车的价格 y:
 tensor([-1.0933, 12.9338,  6.9236], dtype=torch.float64)


In [141]:
# 随机初始化权重向量 w (2,) 和偏置标量 b (1,)
w = torch.randn(2, requires_grad=True, dtype=torch.float64)
b = torch.randn(1, requires_grad=True, dtype=torch.float64)

print("初始盲猜的 w:", w)
print("初始盲猜的 b:", b)


初始盲猜的 w: tensor([0.6394, 0.5293], dtype=torch.float64, requires_grad=True)
初始盲猜的 b: tensor([-0.0732], dtype=torch.float64, requires_grad=True)


In [142]:
# 前向传播：矩阵 X 点乘向量 w，再加上偏置 b
# X @ w 利用了矩阵乘法，一行代码算完 1000 辆车的预测值
y_hat = X @ w + b

# 计算平均均方误差 (MSE)
loss = torch.mean((y_hat - y) ** 2) / 2
print("未训练时的初始 Loss:", loss.item())


未训练时的初始 Loss: 52.45734399404374


In [143]:
lr = 0.0005        # 学习率 (步长)
batch_size = 32  # 每次抓 32 条数据
epochs = 20      # 把 1000 条数据完整看 20 遍

num_samples = len(X)

for epoch in range(epochs):
    # 1. 每一轮开始前，生成随机打乱的索引下标
    indices = torch.randperm(num_samples)
    
    # 2. 按 batch_size 截取数据进行训练
    for i in range(0, num_samples, batch_size):
        # 抓取这一小批的下标
        batch_idx = indices[i : i + batch_size]
        
        X_batch = X[batch_idx] # 形状 (32, 2)
        y_batch = y[batch_idx] # 形状 (32,)
        
        # --- A. 前向传播：计算预测值与 Loss ---
        y_hat = X_batch @ w + b
        loss = torch.mean((y_hat - y_batch) ** 2) / 2
        
        # --- B. 反向传播：自动计算梯度 ---
        loss.backward()
        
        # --- C. 手动更新权重 (梯度下降公式: w = w - lr * grad) ---
        with torch.no_grad(): # 更新参数的过程本身不需要记录计算图！
            w -= lr * w.grad
            b -= lr * b.grad
            
            # ⚠️ 关键：手动清空梯度！否则下一次 backward 会累加！
            w.grad.zero_()
            b.grad.zero_()

    # 每轮打印一下当前学习成果
    with torch.no_grad():
        total_loss = torch.mean(((X @ w + b) - y) ** 2) / 2
        print(f"Epoch {epoch+1:2d} | Loss: {total_loss.item():.4f} | w: [{w[0].item():.2f}, {w[1].item():.2f}] | b: {b.item():.2f}")


Epoch  1 | Loss: 33.2376 | w: [0.29, 0.13] | b: -0.09
Epoch  2 | Loss: 30.0590 | w: [0.16, -0.04] | b: -0.07
Epoch  3 | Loss: 29.4532 | w: [0.12, -0.12] | b: -0.03
Epoch  4 | Loss: 29.2271 | w: [0.11, -0.16] | b: 0.01
Epoch  5 | Loss: 29.0551 | w: [0.10, -0.19] | b: 0.06
Epoch  6 | Loss: 28.9042 | w: [0.11, -0.22] | b: 0.10
Epoch  7 | Loss: 28.7653 | w: [0.11, -0.23] | b: 0.14
Epoch  8 | Loss: 28.6283 | w: [0.11, -0.25] | b: 0.19
Epoch  9 | Loss: 28.5051 | w: [0.11, -0.26] | b: 0.23
Epoch 10 | Loss: 28.3529 | w: [0.12, -0.27] | b: 0.28
Epoch 11 | Loss: 28.2176 | w: [0.12, -0.28] | b: 0.33
Epoch 12 | Loss: 28.0918 | w: [0.12, -0.29] | b: 0.37
Epoch 13 | Loss: 27.9523 | w: [0.12, -0.29] | b: 0.42
Epoch 14 | Loss: 27.8229 | w: [0.13, -0.30] | b: 0.46
Epoch 15 | Loss: 27.6933 | w: [0.13, -0.30] | b: 0.51
Epoch 16 | Loss: 27.5636 | w: [0.14, -0.30] | b: 0.55
Epoch 17 | Loss: 27.4354 | w: [0.13, -0.32] | b: 0.60
Epoch 18 | Loss: 27.3106 | w: [0.14, -0.31] | b: 0.64
Epoch 19 | Loss: 27.1885 |